In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/sidecar_manager.py
import json
import copy
import time
from pathlib import Path
from typing import Any, Dict, Optional, List


class SidecarManager:
    """
    Istanziare una singola volta per filepath (.json).
    La stessa accortezza va realizzata non solo nello stesso notebook,
    ma anche tra notebook DIVERSI.
    Evitare cioè modifiche concorrenti, visto che non c'è nessun lock che le impedisca.
    Nello stesso notebook c'è un WARNING, ma tra notebook diversi non c'è avviso.
    """

    # Tiene traccia dei path per cui è stata creata un istanza.
    #  L'utilizzo di una cache infatti rende problematica la presenza di più istanze:
    #  ciascuna avrebbe una propria cache in RAM indipendente con il rischio che il loro contenuto
    #  diverga senza avvisi.
    #  Un implementazione singleton avrebbe complicato molto il codice, invece tramite
    #  questa accortezza se si istanzia una seconda istanza viene stampato un avviso (non bloccante)
    _seen_paths: set = set()

    def __init__(self, filepath="sidecar_edits.json"):
        # Inizializzazione di default; modificabile passandogli un diverso path come parametro
        self.filepath = (
            Path(filepath).expanduser().resolve()
            if filepath else Path("sidecar_edits.json").resolve()
        )

        # Verifica istanziazione singola
        if self.filepath in SidecarManager._seen_paths:
            print(
                f"[WARNING] Creata una seconda istanza di SidecarManager per lo stesso file "
                f"({self.filepath}). Ogni istanza mantiene una propria cache in RAM: se non e' "
                f"la stessa istanza ad essere condivisa e riutilizzata ovunque, le due cache "
                f"possono disallinearsi silenziosamente. Crea una sola istanza e passala "
                f"esplicitamente a tutti i componenti che leggono/scrivono il sidecar."
            )
        SidecarManager._seen_paths.add(self.filepath)

        # Crea la cache
        self._cache: Optional[Dict[str, Any]] = None

        self._ensure_file_exists()

    def _ensure_file_exists(self):
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
        if not self.filepath.exists():
            self.reset_all()

    ### Accesso e gestione dati e cache
    @property
    def data(self) -> dict:
        """Garantisce l'accesso diretto ai dati leggendoli sempre aggiornati dal file."""
        return self.load_data()

    def load_data(self, force_reload: bool=False) -> dict:
        """
        Carica i dati del sidecaar.
        Sfrutta la cache in RAM per evitare I/O ripetuto e eventuale collo di bottiglia.
        Restituisce sempre la deepcopy, così eventuali mutazioni da parte del chiamante
        (prima di un save_data) non sporcano la cache_interna.
        """
        if self._cache is not None and not force_reload:
            return copy.deepcopy(self._cache)

        if self.filepath.exists():
            try:
                with open(self.filepath, "r", encoding="utf-8") as f:
                    self._cache = json.load(f)
                return copy.deepcopy(self._cache)
            except Exception as e:
                # Il file esiste ma non è leggibile (es. JSON corrotto per
                # un'interruzione a metà scrittura). Prima di scartarlo ne
                # salviamo una copia: la prossima save_data() sovrascriverà
                # il file originale, quindi senza backup il contenuto
                # eventualmente ancora recuperabile andrebbe perso per sempre.
                backup_path = self.filepath.with_suffix(f".corrotto.{int(time.time())}.json")
                try:
                    self.filepath.replace(backup_path)
                    print(f"<[WARNING]> File sidecar illeggibile ({e}). "
                          f"Backup del file originale salvato in: {backup_path}")
                except Exception:
                    print(f"<[WARNING]> File sidecar illeggibile ({e}) e impossibile crearne un backup.")


        default_structure = {
            "pairwise_deltas": {},
            "tag_overrides": {},
            "global_tags": {}
        }
        self._cache = default_structure
        return copy.deepcopy(self._cache)

    def save_data(self, data: dict):
        """Salva il dizionario su disco in modo atomico e aggiorna la cache in RAM."""
        self.filepath.parent.mkdir(parents=True, exist_ok=True)
 
        # Scrittura atomica: si scrive prima su un file temporaneo e poi lo
        # si sostituisce al file definitivo. Path.replace (os.replace) e'
        # atomico su POSIX, quindi anche un'interruzione a meta' (crash,
        # restart del kernel) non puo' lasciare sidecar_edits.json in uno
        # stato troncato/corrotto: resta o la vecchia versione, o gia'
        # completa quella nuova.
        tmp_path = self.filepath.with_suffix(self.filepath.suffix + ".tmp")
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        tmp_path.replace(self.filepath)
 
        self._cache = copy.deepcopy(data)

    def invalidate_cache(self):
        """Forza la rilettura da filesystem alla successiva chiamata di load_data()."""
        self._cache = None
        
    def release_path(self) -> None:
        """
        Rimuove il path di questa istanza dal registro delle istanze viste.
        Da chiamare quando l'istanza non verrà più usata (es. in PreTagger.close()),
        così una nuova SidecarManager sullo stesso file non genera un falso avviso
        di "seconda istanza". NON funziona tra notebook diversi.
        """
        SidecarManager._seen_paths.discard(self.filepath)

    def get_global_tags(self) -> dict:
        """
        Restituisce il dizionario dei tag attivi, con contatore e colore corrente.
        """
        data = self.load_data()
        return data.get("global_tags", {})

    ### Manipolazione Distanze
    def save_pairwise_delta(self, chunk_id_1: str, chunk_id_2: str, distance_factor: float):
        """
        Salva o aggiorna il fattore di distanza tra una coppia di chunk.
        Garantisce la simmetria della relazione (A_B == B_A).
        """
        data = self.load_data()

        # Ordiniamo gli ID per garantire che la relazione sia simmetrica
        pair_key = "_AND_".join(sorted([str(chunk_id_1), str(chunk_id_2)]))

        if "pairwise_deltas" not in data:
            data["pairwise_deltas"] = {}

        data["pairwise_deltas"][pair_key] = {
            "chunk_1": str(chunk_id_1),
            "chunk_2": str(chunk_id_2),
            "distance_factor": round(distance_factor, 3)
        }

        self.save_data(data)
        print(f"<<| Modifica salvata per la coppia [{pair_key}]: factor={distance_factor:.2f} |>>")

    def save_pairwise_deltas_batch(self, edits: list):
        """
        Salva un blocco di modifiche pairwise in un unica operazione I/=
        """
        if not edits or not isinstance(edits, list):
            return

        data = self.load_data()
        if "pairwise_deltas" not in data:
            data["pairwise_deltas"] = {}

        updated = False
        for edit in edits:
            if edit and "chunk_1" in edit and "chunk_2" in edit:
                c1, c2 = str(edit["chunk_1"]), str(edit["chunk_2"])
                pair_key = "_AND_".join(sorted([c1, c2]))
                data["pairwise_deltas"][pair_key] = {
                    "chunk_1": c1,
                    "chunk_2": c2,
                    "distance_factor": round(float(edit["distance_factor"]), 3)
                }
                updated = True

        if updated:
            self.save_data(data)
            print(f"<<| Batch salvato: {len(edits)} distanze aggiornate nel sidecar |>>")


    ### Manipolazione TAG
    DEFAULT_PALETTE = [
        "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab"
    ] # Colori di Default se non ne vengono assegnati altri

    def add_tag_override(self, chunk_id: str, tag: str, color: str=None):
        """
        Aggiunge un tag a un chunk, assegna un colore e aggiorna il registro globale
        """
        data = self.load_data()
        chunk_id = str(chunk_id)

        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}

        if chunk_id not in data["tag_overrides"]:
            data["tag_overrides"][chunk_id] = {"user_tags": []}

        current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])

        # Aggiunge il tag solo se non già presente
        if tag not in current_tags:
            current_tags.append(tag)
            data["tag_overrides"][chunk_id]["user_tags"] = current_tags

            # Aggiorna il registro globale
            if tag not in data["global_tags"]:
                # Se color=None si assegna un colore tra quelli di default
                if not color:
                    used_colors = {
                        t_info.get("color")
                        for t_info in data["global_tags"].values()
                        if isinstance(t_info, dict)
                    }
                    available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                    color = (
                        available[0]
                        if available
                        else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                    ) # Gestione ciclica array
                data["global_tags"][tag] = {"count": 1, "color": color}
            else:
                data["global_tags"][tag]["count"] += 1
                if color: # Aggiornamento colore se esplicitamente fornito
                    data["global_tags"][tag]["color"] = color

            self.save_data(data)
            print(f"<<! Tag '{tag}' ({data['global_tags'][tag]['color']}) aggiunto a [{chunk_id}]. Conteggio globale: {data['global_tags'][tag]['count']}")

    def add_tag_override_for_chunk(self, chunk_id: str, tags: List[str]):
        """
        Assegna una lista di tag a un singolo chunk, aggiornando il registro globale
        ed eseguendo un uica operazione di scrittura su disco per l'intero chunk.
        Da utilizzare ad esempio per il pre-tagging, ottimizzando così l'operazione.
        """
        if not tags:
            return

        data = self.load_data()
        chunk_id = str(chunk_id)

        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}

        if chunk_id not in data["tag_overrides"]:
            data["tag_overrides"][chunk_id] = {"user_tags": []}

        current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])
        updated = False

        for tag in tags:
            if tag not in current_tags:
                current_tags.append(tag)
                updated = True

                if tag not in data["global_tags"]:
                    used_colors = {
                        t_info.get("color")
                        for t_info in data["global_tags"].values()
                        if isinstance(t_info, dict)
                    }
                    available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                    color = (
                        available[0]
                        if available
                        else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                    )
                    data["global_tags"][tag] = {"count": 1, "color": color}
                else:
                    data["global_tags"][tag]["count"] += 1

        if updated:
            data["tag_overrides"][chunk_id]["user_tags"] = current_tags
            self.save_data(data)

    def add_tag_overrides_batch(self, chunk_tags_map: Dict[str, List[str]]) -> None:
        """
        Assegna tag a più chunk in un'unica operazione di lettura+scrittura su disco.
        chunk_tags_map: {chunk_id: [tag1, tag2, ...], ...}
        """
        if not chunk_tags_map:
            return

        data = self.load_data()
        if "tag_overrides" not in data:
            data["tag_overrides"] = {}
        if "global_tags" not in data:
            data["global_tags"] = {}

        updated = False

        for chunk_id, tags in chunk_tags_map.items():
            if not tags:
                continue

            chunk_id = str(chunk_id)

            if chunk_id not in data["tag_overrides"]:
                data["tag_overrides"][chunk_id] = {"user_tags": []}

            # Recupero sicuro di user_tags
            current_tags = data["tag_overrides"][chunk_id].setdefault("user_tags", [])

            for tag in tags:
                if tag not in current_tags:
                    current_tags.append(tag)
                    updated = True

                    if tag not in data["global_tags"]:
                        used_colors = {
                            t.get("color") for t in data["global_tags"].values()
                            if isinstance(t, dict)
                        }
                        available = [c for c in self.DEFAULT_PALETTE if c not in used_colors]
                        color = (
                            available[0] if available
                            else self.DEFAULT_PALETTE[len(data["global_tags"]) % len(self.DEFAULT_PALETTE)]
                        )
                        data["global_tags"][tag] = {"count": 1, "color": color}
                    else:
                        data["global_tags"][tag]["count"] += 1

        if updated:
            self.save_data(data)

    def remove_tag_override(self, chunk_id: str, tag: str):
        """
        Rimuove un tag da un chunk, decrementando il contatore globale.
        Se questo scende a 0, rimuove il tag da tale registro globale.
        """
        data = self.load_data()
        chunk_id = str(chunk_id)

        if "tag_overrides" in data and chunk_id in data["tag_overrides"]:
            current_tags = data["tag_overrides"][chunk_id].get("user_tags", [])
            if tag in current_tags:
                current_tags.remove(tag)
                data["tag_overrides"][chunk_id]["user_tags"] = current_tags

                # Rimuove la voce "tag_overrides" se non presente alcun tag
                if not current_tags:
                    del data["tag_overrides"][chunk_id]

                # Decrementa registro, rimuovendo la voce se necessario
                if "global_tags" in data and tag in data["global_tags"]:
                    data["global_tags"][tag]["count"] -= 1
                    if data["global_tags"][tag]["count"] <= 0:
                        del data["global_tags"][tag]
                        print(f"X>> Tag '{tag}' rimosso definitivamente dal registro globale (conteggio = 0).")
                    else:
                        print(f"<<| Tag '{tag}' rimosso da [{chunk_id}]. Conteggio residuo: {data['global_tags'][tag]['count']}")

                self.save_data(data)

    ### RESET file sidecar
    def reset_all(self):
        """Ripristina il file sidecar azzerando le modifiche (UNDO globale)."""
        self.save_data({
            "pairwise_deltas": {},
            "tag_overrides": {},
            "global_tags": {}
        })
        print("<<! File sidecar ripristinato allo stato iniziale !>>")
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/chunk_widget.py
import sys
import pathlib
from pathlib import Path
import anywidget
import traitlets
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import BASE_GRAPH_DISTANCE
from src.sidecar_manager import SidecarManager

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = pathlib.Path(__file__).parent / "frontend" / "chunk_graph.js"

    # Traitlets per stato del grafo
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
    global_tags = traitlets.Dict({}).tag(sync=True)
    tag_action = traitlets.Dict({}).tag(sync=True)

    #Traitlets per modifiche sulla distanza in batch -- assicura di non perdere edit
    pairwise_edit = traitlets.Dict({}).tag(sync=True)
    pairwise_edits_batch = traitlets.List([]).tag(sync=True)

    def __init__(self, sidecar=None, sidecar_path=None, **kwargs):
        """
        Per evitare di avere una dobbia cache del sidecar, è bene passare il sidecar
        e non il sidecar_path. L'opzione è stata lasciata per test e legacy, ma farlo rischia
        di dare un WARNING che è fondamentale non ignorare se non con coscienza
        """
        super().__init__(**kwargs)
        if isinstance(sidecar, SidecarManager):
            self.sidecar = sidecar
        elif sidecar_path:
            self.sidecar = SidecarManager(filepath=sidecar_path)
        else:
            self.sidecar = SidecarManager()
        self.observe(self._on_pairwise_edit, names=["pairwise_edit"])
        self.observe(self._on_pairwise_edits_batch, names=["pairwise_edits_batch"])
        self.observe(self._on_tag_action, names=["tag_action"])
        self.refresh_global_tags()

    ### Gestione TAG
    def refresh_global_tags(self):
        """Sincronizza il dizionatio dei tag globali con il frontend"""
        self.global_tags = self.sidecar.get_global_tags()

    def _on_tag_action(self, change):
        """Gestisce le azioni di aggiunta e rimozione dei TAG inviate da JavaScript"""
        action_data = change["new"]
        if not action_data:
            return

        action = action_data.get("action")
        chunk_id = action_data.get("chunk_id")
        tag = action_data.get("tag")
        color = action_data.get("color")

        if action == "add" and chunk_id and tag:
            self.sidecar.add_tag_override(chunk_id, tag, color=color)
        elif action == "remove" and chunk_id and tag:
            self.sidecar.remove_tag_override(chunk_id, tag)

        self.refresh_global_tags()
        self._update_node_tags_in_graph_data(chunk_id)

    def _update_node_tags_in_graph_data(self, chunk_id):
        """
        Aggiorna i tag del nodo specifico in graph_data per forzare il ridisegno.
        Semplicemente ricrea e riposiziona i dati (uguali) nel nodo del chunk per forzare il re-render
        """
        data = self.sidecar.load_data()
        chunk_overrides = data.get("tag_overrides", {}).get(str(chunk_id), {})
        user_tags = chunk_overrides.get("user_tags", [])

        new_graph_data = dict(self.graph_data)
        nodes = [dict(n) for n in new_graph_data.get("nodes", [])]
        for n in nodes:
            if str(n.get("id")) == str(chunk_id):
                n["user_tags"] = user_tags
        new_graph_data["nodes"] = nodes
        self.graph_data = new_graph_data

    ### Caricamento grafo
    def load_graph(self, raw_graph_data):
        """Carica il grafo applicando immediatamente i distance_factor salvati nel sidecar."""
        data = self.sidecar.load_data()
        deltas = data.get("pairwise_deltas", {})
        overrides = data.get("tag_overrides", {})

        links = raw_graph_data.get("links", [])
        enriched_links = []

        # archi
        for link in links:
            l_copy = dict(link)
            src = l_copy["source"]["id"] if isinstance(l_copy["source"], dict) else l_copy["source"]
            tgt = l_copy["target"]["id"] if isinstance(l_copy["target"], dict) else l_copy["target"]

            k1 = f"{src}_AND_{tgt}"
            k2 = f"{tgt}_AND_{src}"

            factor = 1.0
            if k1 in deltas:
                factor = deltas[k1].get("distance_factor", 1.0)
            elif k2 in deltas:
                factor = deltas[k2].get("distance_factor", 1.0)

            l_copy["distance_factor"] = factor
            enriched_links.append(l_copy)

        # info sui nodi
        nodes = raw_graph_data.get("nodes", [])
        enriched_nodes = []
        for node in nodes:
            n_copy = dict(node)
            cid = str(n_copy.get("id"))
            if cid in overrides:
                n_copy["user_tags"] = overrides[cid].get("user_tags", [])
            elif "user_tags" not in n_copy:
                n_copy["user_tags"] = n_copy.get("tags", [])
            enriched_nodes.append(n_copy)

        self.graph_data = {
            "nodes": enriched_nodes,
            "links": enriched_links,
            "base_distance": BASE_GRAPH_DISTANCE
        }
        self.refresh_global_tags()

    ### Gestione edit Distanze
    def _on_pairwise_edit(self, change):
        """Gestisce una singola modifica di distanza inviata da JS -- versione meno sicura di pairwise_edits_batch"""
        edit = change["new"]
        if edit and "chunk_1" in edit and "chunk_2" in edit:
            self.sidecar.save_pairwise_delta(
                chunk_id_1=edit["chunk_1"],
                chunk_id_2=edit["chunk_2"],
                distance_factor=edit["distance_factor"]
            )

    def _on_pairwise_edits_batch(self, change):
        """
        Gestisce un blocco (tramite lista) di modifiche simultanee inviato in una sola volta.
        Si verifica quando il trascinamento di 1 nodo altera la distanza con più nodi ad esso collegati.
        Delegata al SidecarManager.
        """
        edits = change["new"]
        self.sidecar.save_pairwise_deltas_batch(edits)
EOF

In [12]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/tag_assigner.py
import re # Per l'analisi lessicale dell'Overlap Coefficient
import sys
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_ollama import ChatOllama
from langid.langid import LanguageIdentifier, model as LANGID_MODEL

from src.config import (
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    EMBEDDING_MODEL,
    LLM_MODEL,
    OLLAMA_URL,
    TAG_ASSIGN_THRESHOLD,
    TAG_WEIGHT_COSINE,
    TAG_WEIGHT_OVERLAP,
)

import numpy as np
from typing import Dict, List, Optional, Union, Set, Tuple

from tqdm import tqdm

# Stopword da escludere nel calcolo dell'Overlap lessicale
#  Serve a impedire che semplici connettivi o articoli gonfino il punteggio
#  di similarità. Copre solo ITA e INGLESE
STOPWORDS: Set[str] = {
    # Italiano
    "il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "di", "a", "da",
    "in", "con", "su", "per", "tra", "fra", "e", "o", "ma", "che", "chi",
    "cui", "non", "più", "del", "dello", "della", "dei", "degli", "delle",
    "al", "allo", "alla", "ai", "agli", "alle", "dal", "dallo", "dalla",
    "dai", "dagli", "dalle", "nel", "nello", "nella", "nei", "negli",
    "nelle", "sul", "sullo", "sulla", "sui", "sugli", "sulle", "questo",
    "questa", "questi", "queste", "quello", "quella", "sono", "sia",
    "stato", "è", "ed", "ad",
    # Inglese
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "up", "about", "into", "through", "after",
    "is", "are", "was", "were", "be", "been", "being", "have", "has",
    "had", "this", "that",
}

# Del fallback di expand_tag(): sono vocaboli di ripiego quando l'espanisone fallisce, vengono quindi eliminati
FALLBACK_TEMPLATE_TOKENS: Set[str] = {"argomenti", "sinonimi", "concetti", "correlati"}

# Per il matching lessicale è necessario che l'espansione dei tag venga effettuata nella lingua del testo fornito
_LANGUAGE_IDENTIFIER = LanguageIdentifier.from_modelstring(LANGID_MODEL, norm_probs=True)
# Soglia sotto cui l'identificazione è inaffidabile. Sotto di essa si ricade sul default value
MIN_CHARS_FOR_LANG_DETECTION = 20
SUPPORTED_LANGUAGES: Set[str] = {"it", "en"} # Le uniche lingue possibilmente riconoscibili

def detect_language(text: str, default: str = "it") -> str:
    """
    Rileva la lingua del testo (codice ISO 639-1) tramite langid. Per testi troppo corti
    la rilevazione non è affidabile o se la lingua non rientra tra quelle supportate, si forza il fallback
    sulla lingua di default.
    """
    if not text or len(text.strip()) < MIN_CHARS_FOR_LANG_DETECTION:
        return default
    
    lang, _confidence = _LANGUAGE_IDENTIFIER.classify(text)
    
    if lang not in SUPPORTED_LANGUAGES:
        return default
        
    return lang

class TagAssigner:
    """
    Assegnatore ibrido deterministico-llm per l'assegnazione dei tag forniti
    inerenti al testo fornito.
    Usato sia nel TagReranker che per il Pre-Tagging, è fatto in modo da poter gestire entrambe le situazioni
    
    Supporta testi in lingue diverse, assegnando correttamente il tag indipendentemente dalle due
    """

    def __init__(
        self,
        threshold: float = TAG_ASSIGN_THRESHOLD,
        weight_cosine: float = TAG_WEIGHT_COSINE,
        weight_overlap: float = TAG_WEIGHT_OVERLAP,
        embedding_model_name: str = EMBEDDING_MODEL,
        llm_model_name: str = LLM_MODEL,
        ollama_url: str = OLLAMA_URL,
        default_language: str = "it",
    ):
        self.threshold = threshold
        self.weight_cosine = weight_cosine
        self.weight_overlap = weight_overlap
        self.default_language = default_language
        
        self.embedding_model = FastEmbedEmbeddings(model_name=embedding_model_name)
        self.llm = ChatOllama(
            model=llm_model_name,
            base_url=ollama_url,
            keep_alive=DEFAULT_KEEP_ALIVE,
            num_thread=DEFAULT_NUM_THREAD,
            temperature=0,
            stop=["\n"], # per evitare format leak nell'espansione del tag
        )

        # Dizionari per memorizzare l'espansione dei tag e i relativi
        #  vettori, risparmiando il ricalcolo ogni volta e permettendo
        #  il riutilizzo degli stessi tag_espansi per tutta l'esecuzione
        #  dipende ovviamente dalla singola istanza della classe, quindi
        #  diviene fondamentale non reistanziarla fintanto che si vuole costanza nelle frasi espansione.
        #  Hanno come chiavi (tag, language): espansione ed embedding cambiano a seconda della lingua del testo analizzato,
        #  ai fini del riconoscimento lessicale (overlap). Quello semantico, basato su cos_sim, si affida alla gestione multilingua dell'embedding model
        self.expanded_tags_cache: Dict[Tuple[str, str], str] = {}
        self.tag_embeddings_cache: Dict[Tuple[str, str], np.ndarray] = {}
        self._fallback_tags: Set[Tuple[str, str]] = set()

    def get_embedding(self, text: str) -> np.ndarray:
        """Restituisce il vettore di embedding"""
        return np.array(self.embedding_model.embed_query(text), dtype=float)

    def expand_tag(self, tag: str, lang: str) -> str:
        """
        Genera una frase descrittiva non ambigua estesa per il tag tramite ChatOllama e salva in cache per riusi futuri.
        Generazione e salvataggio in cache di un tag sono differenziati per lingua del testo analizzato
        """
        cache_key = (tag, lang)
        if cache_key in self.expanded_tags_cache:
            return self.expanded_tags_cache[cache_key]

        prompt = (
            f"Descrivi il tag '{tag}' in lingua '{lang}' in una singola frase densa di informazioni.\n"
            f"Includi obbligatoriamente: sinonimi diretti, sotto-categorie principali, strumenti o componenti chiave, ed entità o termini tecnici correlati.\n"
            f"Evita preamboli da dizionario (es. 'È un'attività che...') e concentrati sull'inserire il maggior numero di sostantivi specifici del settore.\n"
            f"Formato: '{tag}: <frase>'. Senza virgolette."
            """
            f"Fornisci un'espansione descrittiva per il tag '{tag}' in lingua '{lang}'.\n"
            f"Includi concetti generali ampiamente noti, sinonimi e plurali (anche in inglese se diffusi).\n"
            f"Concentrati esclusivamente su sostantivi e concetti chiave, evitando verbi inutili e termini gergali inventati.\n"
            f"Formato tassativo: '{tag}: <testo_espanso>'.\n"
            f"Non usare virgolette e rispondi solo con la riga richiesta."
            """
            """
            f"Genera una singola e breve frase descrittiva per il tag '{tag}', "
            f"scritta ESCLUSIVAMENTE nella lingua con codice ISO 639-1 '{lang}'. "
            f"La frase deve ripetere il tag e dopo : contenere il concetto esteso, sinonimi e termini chiave correlati in quella lingua,"
            f"evitando termini ambigui e inserendo anche i plurali dei termini più significativi e eventuali termini tecnici in inglese e nella lingua del codice ISO. "
            f"Rispondi ESCLUSIVAMENTE con il tag ripetuto seguito dalla frase descrittiva, senza alcun testo aggiuntivo."
            """
        )

        try:
            raw_response = self.llm.invoke(prompt).content.strip()
            if raw_response.startswith(f"{tag}:"): # Per garantire che inizi ripetendo il tag all'inizio
                expanded_text = raw_response
            else:
                expanded_text = f"{tag}: {raw_response}"
        except Exception as e: #frase di fallback
            expanded_text = f"{tag}: argomenti, sinonimi e concetti correlati a {tag}"
            self._fallback_tags.add(cache_key) # Esclusi anche i vocaboli del fallback
            print(f"!!!>>> Errore nell'espansione del tag '{tag}': {e}")

        self.expanded_tags_cache[cache_key] = expanded_text

        ### DEBUG
        print(f"DB>> tag: {tag}")
        print(f"DB>> expanded_tag:\n  >>>{expanded_text}")
        ######
        
        return expanded_text
    
    def cosine_sim(self, v1, v2):
        """Calcola la cosine_similarity tra due vettori"""
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return float(np.dot(v1,v2) / (norm1*norm2))

    def compute_overlap(
        self,
        text: str,
        expanded_tag: str,
        tag: Optional[str] = None,
        lang: Optional[str] = None,
    ) -> float:
        """
        Calcola quanta parte del vocabolario del tag espanso è presente nel testo.
        Si divide intenzionalmente per la sola
        dimensione del tag (differente da "similarità di Jaccard"), per evitare che un testo molto più lungo
        diluisca artificialmente il punteggio. Le stopword vengono escluse da entrambi gli insiemi prima del confronto.
        """
        tokens_text = set(re.findall(r"\b\w+\b", text.lower())) - STOPWORDS
        tokens_tag = set(re.findall(r"\b\w+\b", expanded_tag.lower())) - STOPWORDS
 
        # Se si è ricaduti nel fallback, dal tag esteso si tolgono i vocaboli relativi
        if tag is not None and lang is not None and (tag, lang) in self._fallback_tags:
            tokens_tag -= FALLBACK_TEMPLATE_TOKENS
 
        if not tokens_tag:
            return 0.0
 
        return len(tokens_text.intersection(tokens_tag)) / len(tokens_tag)
 
    
    def assign_tags(
        self,
        text: str,
        candidate_tags: List[str],
        vector: Optional[Union[List[float], np.ndarray]] = None, # rende opzionale, inoltre permette sia ndarray che lista di float
    ) -> List[str]:
        """
        Analizza il testo e restituisce la lista dei soli tag che superano una certa soglia di similarità.
        Lo score comparato con la soglia è calcolato con cos_sim e overlap lessicale pesate tra loro su un tag espanso tramite llm
        Viene individuata la lingua del test per sapere come espandere il tag per la somiglianza lessicale
        """
        lang = detect_language(text, default=self.default_language)
        
        # se passato il vettore usa quello, altrimenti lo calcola
        text_vector = self.get_embedding(text) if vector is None else vector

        assigned_tags = []

        for tag in candidate_tags:            
            expanded_tag_str = self.expand_tag(tag, lang)
            cache_key = (tag, lang)

            if cache_key not in self.tag_embeddings_cache:
                self.tag_embeddings_cache[cache_key] = self.get_embedding(expanded_tag_str)
            tag_vector = self.tag_embeddings_cache[cache_key]
 
            score_cs = self.cosine_sim(text_vector, tag_vector)
 
            score_overlap = self.compute_overlap(text, expanded_tag_str, tag=tag, lang=lang)
 
            final_score = (self.weight_cosine * score_cs) + (self.weight_overlap * score_overlap)

            if final_score >= self.threshold:
                assigned_tags.append(tag)

        return assigned_tags
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/tag_reranker.py
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import numpy as np

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TAG_BOOST_FACTOR, TAG_MALUS_FACTOR, MAX_BOOST, RERANKING_TOP_N
from src.sidecar_manager import SidecarManager
from src.tag_assigner import TagAssigner

class TagReranker:
    """
    Modulo di Reranking che altera lo score di similarità dei chunk candidati
    in base alla corrispondenza tra i tag individuati nella query e i tag dei chunk candidati.

    ---------------------------------------------------------------------------
    Nota di utilizzo consigliata per il Retrieval (Qdrant):
    Si raccomanda di recuperare i chunk candidati da Qdrant tramite:
        response = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_embedded,
            limit=RETRIEVAL_TOP_K,
            with_vectors=True,
            with_payload=True
        )
    L'uso di query_points garantisce l'accesso diretto a hit.score (score CS di base),
    hit.payload (testo e metadati) e hit.vector.
    ---------------------------------------------------------------------------
    """

    def __init__(
        self,
        sidecar_manager: Optional[SidecarManager] = None,
        tag_assigner: Optional[TagAssigner] = None,
        boost_factor: float = TAG_BOOST_FACTOR,
        malus_factor: float = TAG_MALUS_FACTOR,
        max_boost: float = MAX_BOOST
    ):
        self.sidecar_manager = sidecar_manager or SidecarManager()
        self.tag_assigner = tag_assigner or TagAssigner()
        self.boost_factor = boost_factor
        self.malus_factor = malus_factor
        self.max_boost = max_boost

    def rerank(
        self,
        query_text: str,
        retrieved_points: List[Any],
        query_vector: Optional[Union[List[float], np.ndarray]] = None,
        top_n: int=RERANKING_TOP_N
    ) -> List[Dict[str, Any]]:
        """
        Riorordina i punti estratti da Qdrant applicando il moltiplicatore weight_factor.
        """
        if not retrieved_points:
            return []

        # Recupero tag globali attivi
        sidecar_data = self.sidecar_manager.load_data()
        global_tags_dict = sidecar_data.get("global_tags", {})
        candidate_tags = list(global_tags_dict.keys())

        # Assegnazione dei tag alla query e recupero tag dei chunk
        query_tags = []
        if candidate_tags:
            query_tags = self.tag_assigner.assign_tags(
                text=query_text, candidate_tags=candidate_tags, vector=query_vector
            )
        
        tag_overrides = sidecar_data.get("tag_overrides", {})

        # Analisi chunk recuperati e calcolo punteggio finale
        reranked_results = []
        for idx, hit in enumerate(retrieved_points):
            payload = getattr(hit, "payload", {}) or {}
            initial_score = max(0.0, float(hit.score)) # Per sicurezza, così qualunque score non sarà negativo

            # Estrazione ID
            meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload
            doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"
            raw_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
            chunk_idx = raw_idx if raw_idx is not None else idx_global
            chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"

            # TAG del chunk in Analisi
            chunk_info = tag_overrides.get(chunk_id, {})
            chunk_tags = chunk_info.get("user_tags", [])

            # Identificazione n° di tag comuni tra query e chunk
            matched_tags = list(set(query_tags).intersection(set(chunk_tags)))

            # Calcolo weight_factor
            if query_tags and chunk_tags:
                if matched_tags:
                    # MATCH
                    raw_weight= 1.0 + (self.boost_factor * len(matched_tags))
                    weight_factor = min(raw_weight, 1.0 + self.max_boost)
                else:
                    # MISMATCH
                    weight_factor = max(0.1, 1.0 - self.malus_factor)
            else:
                # NIENTE TAG sulla query
                weight_factor = 1.0

            raw_final_score = initial_score * weight_factor

            # Oggetto di output
            result_item = {
                "chunk_id": chunk_id,
                "payload": payload,
                "initial_score": round(initial_score, 4),
                "final_score": round(raw_final_score, 4),
                "_raw_final_score": raw_final_score, # temporanea per il sorting
                "weight_factor": round(weight_factor, 2),
                "query_tags": query_tags,
                "chunk_tags": chunk_tags,
                "matched_tags": matched_tags,
                "vector": getattr(hit, "vector", None),
            }

            reranked_results.append(result_item)
        
        # Ordinamento decrescente
        reranked_results.sort(key=lambda x: x["_raw_final_score"], reverse=True)

        # Rimozione chiave temporanea
        for item in reranked_results:
            del item["_raw_final_score"]

        # Ritorno rei TOP_N dopo il rerank
        if top_n is not None and top_n > 0:
            return reranked_results[:top_n]

        return reranked_results
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/pre_tagger.py
import sys
import logging
from typing import List, Optional, Dict, Union
from pathlib import Path
from qdrant_client import QdrantClient
from tqdm import tqdm #Per barra di avanzamento

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import COLLECTION_NAME, QDRANT_URL
from src.tag_assigner import TagAssigner
from src.sidecar_manager import SidecarManager

import gc # Garbage Collector
import time
import numpy as np

# Per debug e vedere progresso
logger = logging.getLogger(__name__)

class PreTagger:
    """
    Classe per l'esecuzione del pretagging automatico sulla collezione indicata.
    Genera una istanza di SidecarManager per il .json corrispettivo, dunque
    è sempre bene fare il metodo pretagger.close() per eliminare tale istanza
    del sidecar_manager
    """

    def __init__(
        self,
        sidecar_path: str="sidecar_edits.json",
        tag_assigner: Optional[TagAssigner] = None,
        sidecar_manager: Optional[SidecarManager] = None,
        qdrant_client: Optional[QdrantClient] = None,
    ):
        self.sidecar_path = sidecar_path
        
        self.tag_assigner = tag_assigner or TagAssigner()

        # Tiene traccia se sidecar e qdrant sono stati creati qui (e quindi vanno rilasciati
        #  esplicitamente in close()) oppure passati dall'esterno (nel qual caso
        #  resta responsabilità di chi li ha creati).
        self._owns_sidecar_manager = sidecar_manager is None
        self.sidecar_manager = sidecar_manager or SidecarManager(filepath=self.sidecar_path)
        self.sidecar_path = self.sidecar_manager.filepath

        self._owns_qdrant_client = qdrant_client is None
        self.qdrant_client = qdrant_client or QdrantClient(url=QDRANT_URL)
        

    # Per sintassi con "with PreTagger(...) as tagger:"
    #  Garantisce la chiusura se durante run() viene sollevata un eccezione
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

    # Metodi
    def run(
        self,
        candidate_tags: List[str],
        collection_name: str = COLLECTION_NAME,
        batch_size: int = 100,
        start_offset: Optional[Union[int, str]] = None,
        max_chunks: Optional[int] = None,
    ) -> None:
        """
        Scansiona i chunk della collezione e assegna i tag della lista che superano la soglia
        Importando start_offset e max_chunks è possibile delimitare quali chunk della collezione saranno coinvolti
        """
        if not candidate_tags:
            logger.warning("!> Nessun tag candidato fornito")
            return

        logger.info(f"?>> Inizio pre-tagging sulla collezione '{collection_name}' con {len(candidate_tags)} tag")

        # Per barra caricamento (tqdm)
        total_count = None
        try:
            total_count = self.qdrant_client.count(collection_name=collection_name).count
            if max_chunks is not None:
                total_count = min(max_chunks, total_count)
        except Exception as e:
            logger.warning(f"!!!> Impossibile recuperare il conteggio totale dei punti da Qdrant: {e}") 

        pbar = tqdm(total=total_count, desc=f"Pre-tagging '{collection_name}'", unit="chunk")

        next_offset = start_offset # None se il parametro non è passato
        total_processed = 0
        idx_global = 0
        
        while True:
            # Calcolo quanti elementi chiedere nel batch per non superare il max_count
            current_limit = batch_size
            if max_chunks is not None:
                remaining = max_chunks - total_processed
                if remaining <= 0:
                    logger.info(f">> Pre-Taggato l'insieme passato in input")
                    break
                current_limit = min(batch_size, remaining)
            
            # Paginazione su Qdrant
            records, next_offset = self.qdrant_client.scroll(
                collection_name=collection_name,
                limit=current_limit,
                offset=next_offset,
                with_payload=True,
                with_vectors=True,
            )

            if not records:
                logger.info(f">> No records disponibili")
                break

            # Accumulo dei tag assegnati per l'intero batch
            #  Invece di scrivere per ogni chunk, così si ottimizza l'operazione
            page_tags: Dict[str, List[str]] = {}

            for rec in records:
                try:
                    payload = rec.payload or {}
                    chunk_vector = rec.vector
    
                    # Estrazione ID
                    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload
            
                    # Estrazione sicura con fallback protetti da 'or' (evita il bug del valore None)
                    doc_id = payload.get("doc_id") or meta.get("doc_id") or "doc"
                    
                    raw_idx = payload.get("chunk_index") if payload.get("chunk_index") is not None else meta.get("chunk_index")
                    chunk_idx = raw_idx if raw_idx is not None else idx_global
            
                    chunk_id = payload.get("chunk_id") or meta.get("chunk_id") or f"{doc_id}_chunk_{chunk_idx}"
            
                    # Estrazione del testo del chunk
                    chunk_text = payload.get("text") or payload.get("page_content") or meta.get("page_content") or ""
    
                    if chunk_text:
                        # Assegnazione automatica dei tag
                        assigned_tags = self.tag_assigner.assign_tags(
                            text=chunk_text,
                            candidate_tags=candidate_tags,
                            vector=chunk_vector
                        )

                        # Salvataggio in batch dei tag aggiunti
                        if assigned_tags:
                            page_tags[chunk_id] = assigned_tags
        
                except Exception as e:
                    # Se un record specifico fallisce, stampiamo l'errore ed continuiamo col successivo!
                    logger.error(f"!!!> Errore/Blocco sul record all'indice {idx_global} (ID Qdrant: {getattr(rec, 'id', 'sconosciuto')}): {e}")

                total_processed += 1
                idx_global += 1
                pbar.update(1)

            if page_tags:
                try:
                    self.sidecar_manager.add_tag_overrides_batch(page_tags)
                except Exception as e:
                    tqdm.write(f"\n!!!> [ERRORE SCRITTURA SIDECAR] Fallita scrittura batch al chunk {idx_global}: {e}")
                
                page_tags.clear() # Svuota il dizionario per liberare memoria

            # Libera la memoria e rallenta il processo per permettere il flush dell'I/O
            gc.collect()
            time.sleep(0.05)
            
            if next_offset is None:
                break

        pbar.close()
        logger.info(f">> Pre-tagging completato con successo su {total_processed} chunk.")

    def close(self) -> None:
        """
        Rilascia le risorse create da questa istanza. Le risorse iniettate
        dall'esterno (sidecar_manager, qdrant_client) restano responsabilità
        di chi le ha create.
        """
        if hasattr(self, "sidecar_manager"):
            if getattr(self, "_owns_sidecar_manager", False):
                self.sidecar_manager.release_path()
            del self.sidecar_manager

        if hasattr(self, "tag_assigner"):
            del self.tag_assigner

        if hasattr(self, "qdrant_client"):
            if getattr(self, "_owns_qdrant_client", False) and hasattr(self.qdrant_client, "close"):
                try:
                    self.qdrant_client.close()
                except Exception as e:
                    logger.warning(f"!!!> Errore durante la chiusura del client Qdrant: {e}")
            del self.qdrant_client

        logger.info("Istanze interne di PreTagger eliminate dalla memoria.")
EOF